# 04 — Ingestão de Dados Abertos da Receita Federal (CNPJ / QSA)

Este notebook processa a **Base Completa de CNPJs da Receita Federal do Brasil (RFB)**, disponibilizada mensalmente como dado aberto.

## 🎯 Objetivo no Score de Crédito
Cruzar o CPF do cliente com o Quadro de Sócios e Administradores (QSA) para identificar:
- Se é **titular de MEI** ou sócio de empresa ativa
- **Porte da empresa** e capital social envolvido
- **Situação cadastral** (ativa, inapta, baixada)
- **Perfil de atividade econômica** (CNAE) — proxy de setor e risco

> Clientes que são sócios de empresas ativas com capital social relevante tendem a apresentar comportamento financeiro **distinto de trabalhadores CLT ou informais**, sendo um sinal relevante para calibrar limites e risco.

---

## 📦 Arquivos Processados

| Arquivo RFB | Conteúdo | Uso |
|---|---|---|
| `Socios*.zip` | QSA: CPF mascarado, qualificação, data de entrada | **Principal**: vincula cliente à empresa |
| `Empresas*.zip` | CNPJ básico, razão social, porte, capital social | Enriquece info da empresa |
| `Estabelecimentos*.zip` | Situação cadastral, CNAE, município, data abertura | Filtra empresas ativas |
| `Simples*.zip` | Opção pelo Simples Nacional e MEI | Identifica MEI diretamente |

> **⚠️ Volume:** A base completa compactada ultrapassa **7 GB** (dezenas de GB descompactada).  
> Este notebook usa **PySpark** para processamento distribuído no Databricks e salva em **Delta Lake**.

---

## 🔗 Fonte Oficial
```
https://dadosabertos.rfb.gov.br/CNPJ/
```
Atualização: **mensal** (geralmente disponível na primeira semana do mês)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFIGURAÇÕES GERAIS
# ─────────────────────────────────────────────────────────────

# Catálogo Unity Catalog de destino
CATALOG    = "emissor_a"
SCHEMA_RAW = "bronze"
SCHEMA_OUT = "silver"

# Caminho de staging no DBFS/Volumes para os arquivos baixados
STAGING_PATH = f"/Volumes/{CATALOG}/bronze/rfb_cnpj_raw"

# URL base dos dados abertos da RFB
RFB_BASE_URL = "https://dadosabertos.rfb.gov.br/CNPJ/"

# Arquivos a baixar (RFB divide em múltiplas partes numeradas)
# Socios: 1 parte | Empresas: 10 partes | Estabelecimentos: 10 partes | Simples: 1 parte
ARQUIVOS_SOCIOS          = ["Socios0.zip"]
ARQUIVOS_EMPRESAS        = [f"Empresas{i}.zip" for i in range(10)]
ARQUIVOS_ESTABELECIMENTO = [f"Estabelecimentos{i}.zip" for i in range(10)]
ARQUIVOS_SIMPLES         = ["Simples.zip"]

print(f"Destino UC: {CATALOG}.{SCHEMA_OUT}")
print(f"Staging: {STAGING_PATH}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# ETAPA 1: DOWNLOAD DOS ARQUIVOS DA RFB
# Usa dbutils para salvar diretamente no Volume do Unity Catalog
#
# ATENÇÃO: Execute apenas quando precisar atualizar a base.
# O download completo pode levar 20–40 min dependendo da banda.
# Para estudos, baixe apenas Socios0.zip + Simples.zip (~400 MB)
# ─────────────────────────────────────────────────────────────

import subprocess

# Cria o diretório de staging (Volume deve existir no UC)
dbutils.fs.mkdirs(STAGING_PATH)

def download_rfb(arquivo: str, modo_estudo: bool = True):
    """Baixa um arquivo da RFB para o staging no DBFS/Volume."""
    # Em modo_estudo, baixa apenas os arquivos essenciais (Socios + Simples)
    if modo_estudo and arquivo not in ARQUIVOS_SOCIOS + ARQUIVOS_SIMPLES:
        print(f"  [modo estudo] Pulando {arquivo}")
        return

    url    = RFB_BASE_URL + arquivo
    destino = f"/dbfs{STAGING_PATH}/{arquivo}"

    print(f"  ⬇️  Baixando {arquivo}...")
    result = subprocess.run(
        ["wget", "-q", "-O", destino, url],
        capture_output=True, text=True, timeout=3600
    )
    if result.returncode == 0:
        print(f"  ✅  {arquivo} salvo em {destino}")
    else:
        print(f"  ❌  Erro ao baixar {arquivo}: {result.stderr[:200]}")

print("🔄 Iniciando downloads da RFB...\n")
MODO_ESTUDO = True  # ← mude para False para baixar a base completa

for arq in ARQUIVOS_SOCIOS + ARQUIVOS_SIMPLES + ARQUIVOS_EMPRESAS + ARQUIVOS_ESTABELECIMENTO:
    download_rfb(arq, modo_estudo=MODO_ESTUDO)

display(dbutils.fs.ls(STAGING_PATH))

In [ ]:
# ─────────────────────────────────────────────────────────────
# ETAPA 2A: LEITURA DO QSA (SÓCIOS)
# Layout oficial RFB — separador: ; — encoding: latin1
# ─────────────────────────────────────────────────────────────

from pyspark.sql import functions as F
from pyspark.sql.types import *

SCHEMA_SOCIOS = StructType([
    StructField("cnpj_basico",             StringType(), True),
    StructField("identificador_socio",     StringType(), True),  # 1=PJ, 2=PF, 3=Estrangeiro
    StructField("nome_socio",              StringType(), True),
    StructField("cpf_cnpj_socio",          StringType(), True),  # CPF mascarado: ***XXXXXX**
    StructField("cod_qualificacao_socio",  StringType(), True),
    StructField("data_entrada_sociedade",  StringType(), True),  # AAAAMMDD
    StructField("cod_pais",               StringType(), True),
    StructField("cpf_representante_legal", StringType(), True),
    StructField("nome_representante",      StringType(), True),
    StructField("cod_qualificacao_repr",   StringType(), True),
    StructField("faixa_etaria",            StringType(), True),  # 1-9 e 0=N/A
])

df_socios = (
    spark.read
    .option("sep", ";")
    .option("encoding", "latin1")
    .option("header", "false")
    .option("quote", '"')
    .schema(SCHEMA_SOCIOS)
    .csv(f"{STAGING_PATH}/Socios*.csv")  # após descompactação, os .zip viram .csv
)

print(f"QSA — Registros lidos: {df_socios.count():,}")
df_socios.limit(5).display()

In [ ]:
# ─────────────────────────────────────────────────────────────
# ETAPA 2B: LEITURA DE EMPRESAS
# ─────────────────────────────────────────────────────────────

SCHEMA_EMPRESAS = StructType([
    StructField("cnpj_basico",              StringType(), True),
    StructField("razao_social",             StringType(), True),
    StructField("cod_natureza_juridica",    StringType(), True),
    StructField("cod_qualificacao_resp",    StringType(), True),
    StructField("capital_social",           StringType(), True),  # ex: "1000,00"
    StructField("cod_porte_empresa",        StringType(), True),  # 00=N/A,01=ME,03=EPP,05=Demais
    StructField("ente_federativo",          StringType(), True),
])

df_empresas = (
    spark.read
    .option("sep", ";")
    .option("encoding", "latin1")
    .option("header", "false")
    .option("quote", '"')
    .schema(SCHEMA_EMPRESAS)
    .csv(f"{STAGING_PATH}/Empresas*.csv")
)

# Normaliza capital social: "1.000,00" → 1000.0
df_empresas = df_empresas.withColumn(
    "capital_social",
    F.regexp_replace("capital_social", "\\.", "").cast("string")
).withColumn(
    "capital_social",
    F.regexp_replace("capital_social", ",", ".").cast(DoubleType())
)

# Decodifica porte da empresa
PORTE_MAP = {"00": "Nao Aplicavel", "01": "Micro Empresa", "03": "EPP", "05": "Demais"}
df_empresas = df_empresas.replace(PORTE_MAP, subset=["cod_porte_empresa"]).withColumnRenamed(
    "cod_porte_empresa", "porte_empresa"
)

print(f"Empresas — Registros lidos: {df_empresas.count():,}")
df_empresas.limit(3).display()

In [ ]:
# ─────────────────────────────────────────────────────────────
# ETAPA 2C: LEITURA DE ESTABELECIMENTOS (Situação Cadastral + CNAE)
# ─────────────────────────────────────────────────────────────

SCHEMA_ESTAB = StructType([
    StructField("cnpj_basico",           StringType(), True),
    StructField("cnpj_ordem",            StringType(), True),
    StructField("cnpj_dv",              StringType(), True),
    StructField("id_matriz_filial",      StringType(), True),  # 1=Matriz, 2=Filial
    StructField("nome_fantasia",         StringType(), True),
    StructField("cod_situacao_cadastral",StringType(), True),  # 02=Ativa,04=Inapta,08=Baixada
    StructField("data_situacao",         StringType(), True),
    StructField("cod_motivo_situacao",   StringType(), True),
    StructField("cidade_exterior",       StringType(), True),
    StructField("cod_pais",             StringType(), True),
    StructField("data_inicio_atividade", StringType(), True),
    StructField("cnae_principal",        StringType(), True),
    StructField("cnae_secundaria",       StringType(), True),
    StructField("tipo_logradouro",       StringType(), True),
    StructField("logradouro",            StringType(), True),
    StructField("numero",               StringType(), True),
    StructField("complemento",          StringType(), True),
    StructField("bairro",               StringType(), True),
    StructField("cep",                  StringType(), True),
    StructField("uf",                   StringType(), True),
    StructField("cod_municipio",        StringType(), True),
    StructField("ddd1",                 StringType(), True),
    StructField("telefone1",            StringType(), True),
    StructField("ddd2",                 StringType(), True),
    StructField("telefone2",            StringType(), True),
    StructField("ddd_fax",              StringType(), True),
    StructField("fax",                  StringType(), True),
    StructField("email",                StringType(), True),
    StructField("situacao_especial",     StringType(), True),
    StructField("data_situacao_especial",StringType(), True),
])

df_estab = (
    spark.read
    .option("sep", ";")
    .option("encoding", "latin1")
    .option("header", "false")
    .option("quote", '"')
    .schema(SCHEMA_ESTAB)
    .csv(f"{STAGING_PATH}/Estabelecimentos*.csv")
)

# Decodifica situação cadastral
SITUACAO_MAP = {"01": "Nula", "02": "Ativa", "03": "Suspensa", "04": "Inapta", "08": "Baixada"}
df_estab = df_estab.replace(SITUACAO_MAP, subset=["cod_situacao_cadastral"]).withColumnRenamed(
    "cod_situacao_cadastral", "situacao_cadastral"
)

# Seleciona apenas matriz (id_matriz_filial == 1) com colunas relevantes
df_estab_matriz = df_estab.filter(F.col("id_matriz_filial") == "1").select(
    "cnpj_basico", "situacao_cadastral", "data_inicio_atividade",
    "cnae_principal", "uf", "cep"
)

print(f"Estabelecimentos (matriz): {df_estab_matriz.count():,}")
df_estab_matriz.limit(3).display()

In [ ]:
# ─────────────────────────────────────────────────────────────
# ETAPA 2D: LEITURA DO SIMPLES NACIONAL / MEI
# ─────────────────────────────────────────────────────────────

SCHEMA_SIMPLES = StructType([
    StructField("cnpj_basico",         StringType(), True),
    StructField("opcao_simples",       StringType(), True),  # S=Sim, N=Não
    StructField("data_opcao_simples",  StringType(), True),
    StructField("data_exclusao_simples",StringType(), True),
    StructField("opcao_mei",           StringType(), True),  # S=Sim, N=Não
    StructField("data_opcao_mei",      StringType(), True),
    StructField("data_exclusao_mei",   StringType(), True),
])

df_simples = (
    spark.read
    .option("sep", ";")
    .option("encoding", "latin1")
    .option("header", "false")
    .option("quote", '"')
    .schema(SCHEMA_SIMPLES)
    .csv(f"{STAGING_PATH}/Simples.csv")
)

df_simples = df_simples.withColumn(
    "flag_mei",    F.when(F.col("opcao_mei") == "S", 1).otherwise(0)
).withColumn(
    "flag_simples", F.when(F.col("opcao_simples") == "S", 1).otherwise(0)
).select("cnpj_basico", "flag_simples", "flag_mei", "data_opcao_mei")

print(f"Simples — Registros: {df_simples.count():,}")
df_simples.filter("flag_mei = 1").limit(3).display()

In [ ]:
# ─────────────────────────────────────────────────────────────
# ETAPA 3: CONSTRUÇÃO DA VISÃO CONSOLIDADA POR EMPRESA
# Join: Empresas + Estabelecimentos (Matriz) + Simples/MEI
# ─────────────────────────────────────────────────────────────

df_empresa_enriquecida = (
    df_empresas
    .join(df_estab_matriz, on="cnpj_basico", how="left")
    .join(df_simples,      on="cnpj_basico", how="left")
    .fillna({"flag_mei": 0, "flag_simples": 0})
    .withColumn("data_carga",  F.current_timestamp())
    .select(
        "cnpj_basico",
        "razao_social",
        "porte_empresa",
        "capital_social",
        "situacao_cadastral",
        "cnae_principal",
        "uf",
        "flag_simples",
        "flag_mei",
        "data_inicio_atividade",
        "data_carga",
    )
)

print(f"Empresa enriquecida — Registros: {df_empresa_enriquecida.count():,}")
df_empresa_enriquecida.filter("flag_mei = 1").limit(5).display()

In [ ]:
# ─────────────────────────────────────────────────────────────
# ETAPA 4: CONSTRUÇÃO DO PERFIL SOCIETÁRIO DO CLIENTE (QSA)
# Objetivo: uma linha por CPF mascarado com agregações
#
# ⚠️  LGPD: O CPF vem mascarado (***XXXXXX**) pela RFB.
# O cruzamento com dados de clientes deve ocorrer em ambiente
# seguro com chave interna — nunca expor CPF em claro.
# ─────────────────────────────────────────────────────────────

# Decodifica faixa etária
FAIXA_ETARIA_MAP = {
    "0": "Nao Aplicavel", "1": "0-12",  "2": "13-20",
    "3": "21-30",         "4": "31-40", "5": "41-50",
    "6": "51-60",         "7": "61-70", "8": "71-80", "9": "80+"
}

df_qsa_pf = (
    df_socios
    # Filtra apenas Pessoas Físicas (identificador_socio == '2')
    .filter(F.col("identificador_socio") == "2")
    .replace(FAIXA_ETARIA_MAP, subset=["faixa_etaria"])
    # Join com empresa enriquecida
    .join(df_empresa_enriquecida, on="cnpj_basico", how="left")
)

# Agrega por CPF mascarado: perfil consolidado de vínculo societário
df_perfil_socio = (
    df_qsa_pf
    .groupBy("cpf_cnpj_socio", "nome_socio", "faixa_etaria")
    .agg(
        F.count("*")                                         .alias("qtd_vinculos_societarios"),
        F.sum("flag_mei")                                    .alias("qtd_empresas_mei"),
        F.sum("flag_simples")                                .alias("qtd_empresas_simples"),
        F.sum(F.when(F.col("situacao_cadastral") == "Ativa", 1).otherwise(0))
                                                             .alias("qtd_empresas_ativas"),
        F.max("capital_social")                              .alias("maior_capital_social"),
        F.sum("capital_social")                              .alias("capital_social_total"),
        F.collect_set("porte_empresa")                       .alias("portes_empresa"),
        F.collect_set("situacao_cadastral")                  .alias("situacoes_cadastrais"),
        F.collect_set("cnae_principal")                      .alias("cnaes_principais"),
        F.min("data_inicio_atividade")                       .alias("data_primeiro_cnpj"),
    )
    # Features booleanas derivadas
    .withColumn("flag_possui_cnpj",      F.lit(1))
    .withColumn("flag_tem_empresa_ativa",F.when(F.col("qtd_empresas_ativas") > 0, 1).otherwise(0))
    .withColumn("flag_tem_mei",          F.when(F.col("qtd_empresas_mei") > 0, 1).otherwise(0))
    .withColumn("flag_multi_empresas",   F.when(F.col("qtd_vinculos_societarios") > 1, 1).otherwise(0))
    .withColumn("data_carga",            F.current_timestamp())
)

print(f"Perfis societários (CPFs únicos): {df_perfil_socio.count():,}")
df_perfil_socio.limit(5).display()

In [ ]:
# ─────────────────────────────────────────────────────────────
# ETAPA 5: ANÁLISE EXPLORATÓRIA
# ─────────────────────────────────────────────────────────────

print("=== Distribuição por Porte de Empresa ===")
df_qsa_pf.groupBy("porte_empresa").count().orderBy("count", ascending=False).display()

print("\n=== Distribuição por Situação Cadastral ===")
df_qsa_pf.groupBy("situacao_cadastral").count().orderBy("count", ascending=False).display()

print("\n=== Top 10 CNAEs mais comuns entre sócios PF ===")
df_qsa_pf.groupBy("cnae_principal").count().orderBy("count", ascending=False).limit(10).display()

print("\n=== Distribuição de Faixa Etária dos Sócios PF ===")
df_qsa_pf.groupBy("faixa_etaria").count().orderBy("faixa_etaria").display()

print("\n=== % de MEIs entre todos os vínculos de PF ===")
total = df_qsa_pf.count()
mei   = df_qsa_pf.filter("flag_mei = 1").count()
print(f"MEI: {mei:,} de {total:,} ({mei/total*100:.1f}%)")

In [ ]:
# ─────────────────────────────────────────────────────────────
# ETAPA 6: PERSISTÊNCIA NO UNITY CATALOG
# Salva 3 tabelas no schema silver:
#   - rfb_empresa_enriquecida
#   - rfb_qsa_pf_detalhado
#   - rfb_perfil_socio (feature store)
# ─────────────────────────────────────────────────────────────

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_OUT}")

tabelas = {
    "rfb_empresa_enriquecida": df_empresa_enriquecida,
    "rfb_qsa_pf_detalhado":    df_qsa_pf,
    "rfb_perfil_socio":        df_perfil_socio,
}

for nome_tabela, df in tabelas.items():
    full_name = f"{CATALOG}.{SCHEMA_OUT}.{nome_tabela}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("delta.autoOptimize.optimizeWrite", "true")  # Databricks: compacta automaticamente
        .saveAsTable(full_name)
    )
    print(f"  ✅ Tabela salva: {full_name}")

print("\n✅ Carga completa!")

In [ ]:
# ─────────────────────────────────────────────────────────────
# ETAPA 7: VALIDAÇÃO PÓS-CARGA
# ─────────────────────────────────────────────────────────────

for nome_tabela in tabelas.keys():
    full_name = f"{CATALOG}.{SCHEMA_OUT}.{nome_tabela}"
    cnt = spark.sql(f"SELECT COUNT(*) AS total FROM {full_name}").collect()[0]["total"]
    print(f"  📊 {full_name}: {cnt:,} registros")

print("\n=== Amostra da Feature Store (rfb_perfil_socio) ===")
spark.sql(f"""
    SELECT
        cpf_cnpj_socio,
        qtd_vinculos_societarios,
        qtd_empresas_ativas,
        flag_tem_mei,
        capital_social_total,
        portes_empresa
    FROM {CATALOG}.{SCHEMA_OUT}.rfb_perfil_socio
    WHERE flag_tem_empresa_ativa = 1
    LIMIT 10
""").display()

## ✅ Próximos Passos

As tabelas estão disponíveis no Unity Catalog para:

### 1. Join com dados de clientes em `02_feature_engineering/`
```sql
-- Cruza pelo CPF mascarado (chave de matching)
SELECT
    c.*,
    COALESCE(s.flag_possui_cnpj,    0) AS flag_possui_cnpj,
    COALESCE(s.flag_tem_empresa_ativa, 0) AS flag_tem_empresa_ativa,
    COALESCE(s.flag_tem_mei,         0) AS flag_tem_mei,
    COALESCE(s.qtd_empresas_ativas,  0) AS qtd_empresas_ativas,
    COALESCE(s.capital_social_total, 0) AS capital_social_total,
    COALESCE(s.flag_multi_empresas,  0) AS flag_multi_empresas
FROM emissor_a.gold.clientes         c
LEFT JOIN emissor_a.silver.rfb_perfil_socio s
  ON c.cpf_mascarado = s.cpf_cnpj_socio
```

### 2. Features derivadas para o modelo
| Feature | Sinal esperado |
|---|---|
| `flag_tem_empresa_ativa = 1` | Menor risco (empresário ativo) |
| `flag_tem_mei = 1` | Risco intermediário (autônomo formalizado) |
| `capital_social_total > X` | Menor risco (capacidade financeira proxy) |
| `flag_multi_empresas = 1` | Depende do setor e situação cadastral |
| `situacoes_cadastrais` contém 'Baixada' | Maior risco |

### 3. Atualização mensal
Agende este notebook via **Databricks Workflows** para rodar na primeira semana de cada mês, quando a RFB publica a nova versão da base.